# `05_qa_reconciliation.ipynb` — bridge s06 detections to Ken's QA decisions

The `parcel_development_history_etl/steps/s06_qa.py` ETL step writes 8 reconciliation CSVs to `data/qa_data/` flagging **what looks wrong** about the data. Ken's `CA Changes breakdown.xlsx` records **what was decided** about errors during the 2023 and 2026 big-sweep campaigns.

This notebook joins them. For every s06 finding, output a label:

- `addressed_2023`        — Ken corrected this APN during the 2023 big sweep
- `addressed_2026`        — Ken corrected this APN during the 2026 big sweep
- `addressed_both`        — corrected in both cycles
- `pending`               — s06 flagged it, Ken hasn't addressed yet

Plus the reverse direction: any APNs Ken corrected that s06 didn't flag (`ken_only_corrections` — could be data-quality concern or normal pre-detection workflow).

Inputs:
- `data/qa_data/qa_change_events.csv` (from `04_load_ca_changes.ipynb`)
- `data/qa_data/QA_Lost_APNs.csv` (s06)
- `data/qa_data/QA_FC_Units_Not_In_CSV.csv` (s06)
- `data/qa_data/QA_Unit_Reconciliation.csv` (s06)

Outputs:
- `data/qa_data/qa_reconciliation_status.csv` — one row per s06 finding, labeled
- `data/qa_data/qa_ken_only_corrections.csv` — Ken's events with no matching s06 finding

Kernel: `arcgispro-py3`. Read-only against inputs.

## 1. Imports + paths + APN canonicalization

In [1]:
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path
import re
import pandas as pd

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
QA_DIR = REPO_ROOT / 'data' / 'qa_data'

S06_LOST     = QA_DIR / 'QA_Lost_APNs.csv'
S06_FC_GAP   = QA_DIR / 'QA_FC_Units_Not_In_CSV.csv'
S06_RECONCIL = QA_DIR / 'QA_Unit_Reconciliation.csv'
KEN_EVENTS   = QA_DIR / 'qa_change_events.csv'

OUT_STATUS   = QA_DIR / 'qa_reconciliation_status.csv'
OUT_KEN_ONLY = QA_DIR / 'qa_ken_only_corrections.csv'

LOADED_AT = datetime.now(timezone.utc).isoformat()

# Mirror the canonicalization from 04_load_ca_changes.ipynb
_STD_APN_RE = re.compile(r'^(\d{3})-(\d{3})-(\d{2,3})$')
def canonical_apn(raw):
    if not isinstance(raw, str) or not raw.strip():
        return None
    s = raw.strip()
    m = _STD_APN_RE.match(s)
    if m:
        return f'{m.group(1)}-{m.group(2)}-{m.group(3).zfill(3)}'
    return s

for p in [S06_LOST, S06_FC_GAP, S06_RECONCIL, KEN_EVENTS]:
    print(f'{"OK" if p.exists() else "MISSING":7} {p}')

OK      C:\Users\mbindl\Documents\GitHub\Reporting\data\qa_data\QA_Lost_APNs.csv
OK      C:\Users\mbindl\Documents\GitHub\Reporting\data\qa_data\QA_FC_Units_Not_In_CSV.csv
OK      C:\Users\mbindl\Documents\GitHub\Reporting\data\qa_data\QA_Unit_Reconciliation.csv
OK      C:\Users\mbindl\Documents\GitHub\Reporting\data\qa_data\qa_change_events.csv


## 2. Load Ken's events into a lookup index

Build `ken_index: dict[CanonicalAPN] -> set[ReportingYear]` for O(1) lookup.

In [2]:
ken = pd.read_csv(KEN_EVENTS)
print(f'Ken events: {len(ken):,}')

ken_index = {}
for _, r in ken.iterrows():
    apn = r['CanonicalAPN']
    yr = r['Year']
    ken_index.setdefault(apn, set()).add(int(yr))

print(f'Unique CanonicalAPNs Ken touched: {len(ken_index):,}')
print(f'  ...with 2023 corrections: {sum(1 for v in ken_index.values() if 2023 in v):,}')
print(f'  ...with 2026 corrections: {sum(1 for v in ken_index.values() if 2026 in v):,}')
print(f'  ...with both:             {sum(1 for v in ken_index.values() if 2023 in v and 2026 in v):,}')

Ken events: 5,925
Unique CanonicalAPNs Ken touched: 4,903
  ...with 2023 corrections: 3,863
  ...with 2026 corrections: 2,061
  ...with both:             1,021


## 3. Label each s06 finding

Three input CSVs, each with an APN. Apply the same canonicalization, then look up Ken's events.

In [3]:
def label(canon_apn):
    """Return one of: addressed_2023 | addressed_2026 | addressed_both | pending"""
    if canon_apn not in ken_index:
        return 'pending'
    yrs = ken_index[canon_apn]
    has23, has26 = 2023 in yrs, 2026 in yrs
    if has23 and has26: return 'addressed_both'
    if has23:           return 'addressed_2023'
    if has26:           return 'addressed_2026'
    return 'pending'

def load_and_label(path, source_name, has_year=True):
    df = pd.read_csv(path)
    df['CanonicalAPN'] = df['APN'].apply(canonical_apn)
    df['Status'] = df['CanonicalAPN'].apply(label)
    df['s06_source'] = source_name
    keep = ['s06_source', 'CanonicalAPN', 'APN', 'Status']
    if has_year and 'Year' in df.columns:
        keep.append('Year')
    keep += [c for c in df.columns if c not in keep and c != 'CanonicalAPN' and c != 'Status']
    return df[keep]

lost   = load_and_label(S06_LOST,     'QA_Lost_APNs',          has_year=False)
fc_gap = load_and_label(S06_FC_GAP,   'QA_FC_Units_Not_In_CSV', has_year=True)
recon  = load_and_label(S06_RECONCIL, 'QA_Unit_Reconciliation', has_year=True)

status = pd.concat([lost, fc_gap, recon], ignore_index=True, sort=False)
print(f'Total s06 findings labeled: {len(status):,}')

Total s06 findings labeled: 218,192


## 4. Status breakdown per s06 source

In [4]:
summary = status.groupby(['s06_source', 'Status']).size().unstack(fill_value=0)
summary['Total'] = summary.sum(axis=1)
for col in ['addressed_2023', 'addressed_2026', 'addressed_both', 'pending']:
    if col not in summary.columns:
        summary[col] = 0
ordered = ['addressed_2023', 'addressed_2026', 'addressed_both', 'pending', 'Total']
summary = summary[ordered]
summary['% addressed'] = ((summary['addressed_2023'] + summary['addressed_2026'] + summary['addressed_both'])
                          / summary['Total'] * 100).round(1)
print('=== s06 findings by source x Ken status ===')
print(summary.to_string())

=== s06 findings by source x Ken status ===
Status                  addressed_2023  addressed_2026  addressed_both  pending   Total  % addressed
s06_source                                                                                          
QA_FC_Units_Not_In_CSV            3512            3739            1749   148820  157820          5.7
QA_Lost_APNs                       164             137             179      218     698         68.8
QA_Unit_Reconciliation            1899            7175            6731    43869   59674         26.5


## 5. Reverse direction — Ken corrections without a matching s06 finding

Build the inverse: which APNs did Ken correct that s06 *didn't* flag? These could be:

- Pre-detection corrections (Ken caught issues before s06 ran)
- s06 false negatives (real issues s06 misses)
- Workflow noise (corrections to data outside s06's scope)

In [5]:
s06_apns = set(status['CanonicalAPN'].dropna().unique())
ken_only_apns = set(ken_index.keys()) - s06_apns
print(f's06 unique APNs flagged: {len(s06_apns):,}')
print(f'Ken APNs corrected:      {len(ken_index):,}')
print(f'Ken-only (no s06 match): {len(ken_only_apns):,}')

ken_only = ken[ken['CanonicalAPN'].isin(ken_only_apns)].copy()
ken_only['ReconciliationNote'] = 'no_matching_s06_finding'
print(f'\nKen-only event rows: {len(ken_only):,}')
if len(ken_only):
    print('  by Year:')
    print(ken_only.groupby('Year').size().to_string())

s06 unique APNs flagged: 43,774
Ken APNs corrected:      4,903
Ken-only (no s06 match): 1,031

Ken-only event rows: 1,035
  by Year:
Year
2023    1031
2026       4


## 6. Write outputs

In [6]:
status['LoadedAt'] = LOADED_AT
status.to_csv(OUT_STATUS, index=False)
print(f'Wrote {OUT_STATUS.name}: {len(status):,} rows')

ken_only['LoadedAt'] = LOADED_AT
ken_only.to_csv(OUT_KEN_ONLY, index=False)
print(f'Wrote {OUT_KEN_ONLY.name}: {len(ken_only):,} rows')

print('\n=== Headline numbers ===')
tot = len(status)
addressed = (status['Status'] != 'pending').sum()
pending = (status['Status'] == 'pending').sum()
print(f's06 findings TOTAL:       {tot:,}')
print(f's06 findings ADDRESSED:   {addressed:,} ({addressed/tot*100:.1f}%)')
print(f's06 findings PENDING:     {pending:,} ({pending/tot*100:.1f}%)')
print(f'Ken-only corrections:     {len(ken_only):,}')

Wrote qa_reconciliation_status.csv: 218,192 rows
Wrote qa_ken_only_corrections.csv: 1,035 rows

=== Headline numbers ===
s06 findings TOTAL:       218,192
s06 findings ADDRESSED:   25,285 (11.6%)
s06 findings PENDING:     192,907 (88.4%)
Ken-only corrections:     1,035


## Next steps

1. **Dashboard E1** — feed `qa_reconciliation_status.csv` into proposed_dashboards.md cluster E1 (Change Rationale Audit Trail). AG Grid table grouped by `s06_source` × `Status`, filterable by Year/COUNTY/Category. Drill: click a `pending` row to open the s06 finding's detail.
2. **Year-aware matching** — current logic matches APN-only; for `QA_FC_Units_Not_In_CSV` and `QA_Unit_Reconciliation`, also confirm the Year falls within Ken's correction context (e.g., a 2023 correction shouldn't "address" a 2024 detection).
3. **Promote to script** — once stable, fold both `04_load_ca_changes.ipynb` and `05_qa_reconciliation.ipynb` into `parcel_development_history_etl/steps/s07_load_and_reconcile_ca_changes.py`.